# 🧪 BabyChatty RAG System — Evaluation Notebook

**UC3M · NLP Final Project 2025/2026**

This notebook systematically evaluates the BabyChatty RAG pipeline across two key dimensions:

1. **Parameter Validation** — Justifying `k` (top-K retrieval), temperature, and chunk size choices
2. **Embedding Model Comparison** — `BAAI/bge-m3` vs `all-MiniLM-L6-v2`

**Metrics evaluated (per project rubric Table 1):**
- 📚 Document Coverage — % of in-scope queries answered
- ⏱️ Response Time — seconds per query
- 🧠 RAG System Quality — Faithfulness, Answer Relevancy, Context Utilization (via RAGAS)
- 🔴 Hallucination Flag — False Positives (answered when should not)
- 📉 Classification metrics — Accuracy, Precision, Recall, F1

---

## 0. Setup & Imports

In [ ]:
import os, sys, time, re, csv, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
load_dotenv()

# ── Add project root to path ─────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent   # adjust if notebook is inside /notebooks
sys.path.insert(0, str(PROJECT_ROOT))

# ── Plotting style ───────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = {
    'bge':    '#4db8b0',   # teal  → bge-m3
    'minilm': '#f4845f',   # coral → all-MiniLM-L6-v2
    'accent': '#9b7fca',   # purple
    'ok':     '#4caf50',
    'warn':   '#e86c47',
}

print('✅ Imports OK')
print(f'   Project root: {PROJECT_ROOT}')

In [ ]:
# ── Core project imports ─────────────────────────────────────────────────────
import ollama
from langdetect import detect, LangDetectException
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, ContextUtilization
from datasets import Dataset

from utils import GenConfig as cfg, VectorDBFactory

print('✅ Project modules loaded')
print(f'   Main model  : {cfg.model_name}')
print(f'   Judge model : {cfg.judge_name}')
print(f'   Embedding   : {cfg.embedding_model}')
print(f'   Ollama URL  : {cfg.ollama_url}')

## 1. Load Evaluation Dataset

The dataset (`eval_questions_v2.csv`) contains **105 questions** across four categories:

| Category | Description | Label |
|---|---|---|
| `answerable` (EN) | Pediatric health questions in English | 1 |
| `answerable` (ES) | Pediatric health questions in Spanish | 1 |
| `not_answerable` (EN) | Out-of-scope questions in English | 0 |
| `not_answerable` (ES) | Out-of-scope questions in Spanish | 0 |

In [ ]:
EVAL_CSV = PROJECT_ROOT / 'data' / 'eval_questions_v2.csv'
df_eval = pd.read_csv(EVAL_CSV)

print(f'Total questions: {len(df_eval)}')
print(df_eval.groupby(['category', 'language', 'label']).size().reset_index(name='count').to_string(index=False))
df_eval.head(10)

## 2. Helper Functions

All evaluation logic is self-contained here so each experiment section can run independently.

In [ ]:
# ── Ollama client (shared across all experiments) ────────────────────────────
client = ollama.Client(
    host=cfg.ollama_url,
    headers={'X-API-KEY': os.getenv('OLLAMA_API_KEY')},
)

# ── RAGAS judge LLM ──────────────────────────────────────────────────────────
eval_llm = ChatOpenAI(
    base_url=f"{cfg.ollama_url}/v1",
    api_key='fake-key',
    model=cfg.judge_name,
    default_headers={'X-API-KEY': os.getenv('OLLAMA_API_KEY')},
    temperature=0,
    timeout=120,
)

# ── Language detection ───────────────────────────────────────────────────────
def detect_language(text: str) -> str:
    lang_map = {'es': 'Spanish', 'en': 'English', 'fr': 'French',
                'de': 'German', 'it': 'Italian', 'pt': 'Portuguese', 'ca': 'Catalan'}
    try:
        return lang_map.get(detect(text), 'English')
    except LangDetectException:
        return 'English'

# ── Query enrichment (adds pediatric context for retrieval) ──────────────────
PEDIATRIC_KW = ['baby','infant','toddler','child','children','newborn',
                'bebé','niño','niña','lactante','pediatr']
def enrich_query(text: str) -> str:
    if any(kw in text.lower() for kw in PEDIATRIC_KW):
        return text
    return f'baby infant child pediatric: {text}'

# ── Negative-answer detection ────────────────────────────────────────────────
NO_INFO_PATTERNS = cfg.no_info_patterns

def is_negative(answer: str, docs: list) -> bool:
    if not docs:
        return True
    return any(re.search(p, answer.lower()) for p in NO_INFO_PATTERNS)

# ── Single RAG turn ──────────────────────────────────────────────────────────
def run_rag_turn(question: str, vectordb, k: int, temperature: float) -> dict:
    """Run one full RAG cycle. Returns a result dict."""
    t0 = time.time()
    lang = detect_language(question)
    enriched = enrich_query(question)

    docs = vectordb.similarity_search(enriched, k=k)

    context_parts = []
    for i, doc in enumerate(docs, 1):
        title  = doc.metadata.get('title',  'Unknown')
        source = doc.metadata.get('source', 'Unknown')
        context_parts.append(f'[Source {i}: {title} | {source}]\n{doc.page_content}')
    context = '\n\n'.join(context_parts)

    prompt = f"""You are a professional Pediatric Assistant.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say you don't know.
If a completely different topic is asked (weather, sports, cooking, etc.), say you don't have that information.
Never invent any response; if it is not in the data, say you don't know.
ALL health questions must be interpreted in a pediatric context.
Before answering, check internally whether the context is sufficient. Do not show your reasoning.

CRITICAL LANGUAGE RULE: The user is writing in {lang}. Respond entirely in {lang}.

Context: {context}

Question: {question}
Helpful Answer in {lang}:"""

    response = client.chat(
        model=cfg.model_name,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    answer = response['message']['content']
    elapsed = time.time() - t0

    negative = is_negative(answer, docs)
    return {
        'answer':   answer,
        'docs':     docs,
        'lang':     lang,
        'time':     elapsed,
        'negative': negative,
    }

# ── RAGAS evaluation for one turn ────────────────────────────────────────────
def ragas_eval(question: str, answer: str, docs: list,
               embeddings_model) -> dict:
    """Run RAGAS on a single Q/A/context triple. Returns metric dict."""
    contexts = [d.page_content for d in docs]
    dataset  = Dataset.from_dict({
        'question': [question],
        'answer':   [answer],
        'contexts': [contexts],
    })
    try:
        results = evaluate(
            dataset=dataset,
            metrics=[faithfulness, answer_relevancy, ContextUtilization()],
            llm=eval_llm,
            embeddings=embeddings_model,
            show_progress=False,
        )
        df_r = results.to_pandas()
        return {
            'faithfulness':        float(df_r['faithfulness'].iloc[0]),
            'answer_relevancy':    float(df_r['answer_relevancy'].iloc[0]),
            'context_utilization': float(df_r['context_utilization'].iloc[0]),
        }
    except Exception as e:
        print(f'  [RAGAS error] {e}')
        return {'faithfulness': np.nan, 'answer_relevancy': np.nan,
                'context_utilization': np.nan}

# ── Full batch evaluation ─────────────────────────────────────────────────────
def run_experiment(questions_df: pd.DataFrame, vectordb,
                   embeddings_model, k: int, temperature: float,
                   run_ragas: bool = True, sample_n: int = None,
                   seed: int = 42) -> pd.DataFrame:
    """
    Run the full evaluation pipeline on a question dataframe.
    Returns a results dataframe with one row per question.
    """
    if sample_n:
        questions_df = questions_df.sample(n=sample_n, random_state=seed).reset_index(drop=True)

    rows = []
    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df),
                       desc=f'k={k} temp={temperature}'):
        q     = row['question']
        label = int(row['label'])
        qid   = row.get('id', '')
        lang_col  = row.get('language', '')
        cat_col   = row.get('category', '')

        result = run_rag_turn(q, vectordb, k=k, temperature=temperature)
        pred   = 0 if result['negative'] else 1

        ragas_scores = {'faithfulness': np.nan, 'answer_relevancy': np.nan,
                        'context_utilization': np.nan}
        if run_ragas and not result['negative']:
            ragas_scores = ragas_eval(q, result['answer'],
                                      result['docs'], embeddings_model)

        # Hallucination = model answered (pred=1) but label=0 (not answerable)
        hallucination = int(pred == 1 and label == 0)

        rows.append({
            'id':                   qid,
            'question':             q,
            'language':             lang_col,
            'category':             cat_col,
            'label':                label,
            'pred':                 pred,
            'response_time':        result['time'],
            'hallucination_flag':   hallucination,
            'faithfulness':         ragas_scores['faithfulness'],
            'answer_relevancy':     ragas_scores['answer_relevancy'],
            'context_utilization':  ragas_scores['context_utilization'],
            'answer_preview':       result['answer'][:200],
        })

    return pd.DataFrame(rows)

# ── Summary metrics ───────────────────────────────────────────────────────────
def summarize(df: pd.DataFrame, label: str) -> dict:
    """Compute aggregate metrics from a results dataframe."""
    total   = len(df)
    pos     = df[df['label'] == 1]
    neg     = df[df['label'] == 0]
    TP = int(((df['label']==1) & (df['pred']==1)).sum())
    FP = int(((df['label']==0) & (df['pred']==1)).sum())
    TN = int(((df['label']==0) & (df['pred']==0)).sum())
    FN = int(((df['label']==1) & (df['pred']==0)).sum())

    accuracy  = (TP+TN)/total if total else 0
    precision = TP/(TP+FP)    if (TP+FP) else 0
    recall    = TP/(TP+FN)    if (TP+FN) else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    coverage  = TP/(TP+FN)    if (TP+FN) else 0   # recall on positives
    hall_rate = FP/(FP+TN)    if (FP+TN) else 0

    def safe_mean(col):
        vals = df[col].dropna()
        return float(vals.mean()) if len(vals) else np.nan

    faith = safe_mean('faithfulness')
    rel   = safe_mean('answer_relevancy')
    ctx   = safe_mean('context_utilization')
    quality = np.nanmean([faith, rel, ctx])

    return {
        'label':                label,
        'n_total':              total,
        'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
        'accuracy':             round(accuracy,  3),
        'precision':            round(precision, 3),
        'recall':               round(recall,    3),
        'f1':                   round(f1,        3),
        'document_coverage':    round(coverage,  3),
        'hallucination_rate':   round(hall_rate, 3),
        'avg_response_time_s':  round(df['response_time'].mean(), 2),
        'faithfulness':         round(faith, 3) if not np.isnan(faith) else np.nan,
        'answer_relevancy':     round(rel,   3) if not np.isnan(rel)   else np.nan,
        'context_utilization':  round(ctx,   3) if not np.isnan(ctx)   else np.nan,
        'rag_system_quality':   round(quality,3) if not np.isnan(quality) else np.nan,
    }

print('✅ Helper functions defined')

---
## 3. Experiment A — Parameter Validation: Top-K Retrieval

**Hypothesis:** Increasing `k` provides more context but may introduce noise, reducing faithfulness. We test `k ∈ {3, 5, 7, 10}` with fixed temperature=0 and the BGE-M3 embedding model.

> ⏱️ This section runs the model. Expected time: ~5–10 min depending on server load.

In [ ]:
# ── Load the production vector DB (BGE-M3, already built) ────────────────────
print('Loading BGE-M3 vector database...')
emb_bge = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': cfg.emb_device}
)
vectordb_bge = Chroma(
    persist_directory=str(cfg.chroma_dir),
    embedding_function=emb_bge
)
print(f'  ✅ DB loaded — collection size: {vectordb_bge._collection.count()} chunks')

In [ ]:
# ── Sample a balanced subset for k experiments (faster) ──────────────────────
# Use 20 answerable + 10 not_answerable = 30 questions
sample_ans = df_eval[df_eval['label']==1].sample(n=20, random_state=42)
sample_not = df_eval[df_eval['label']==0].sample(n=10, random_state=42)
df_k_sample = pd.concat([sample_ans, sample_not]).reset_index(drop=True)

K_VALUES = [3, 5, 7, 10]
results_k = {}

for k in K_VALUES:
    print(f'\n── Running k={k} ─────────────────────────────────────────')
    df_res = run_experiment(
        df_k_sample, vectordb_bge, emb_bge,
        k=k, temperature=0.0,
        run_ragas=True
    )
    results_k[k] = df_res
    s = summarize(df_res, label=f'k={k}')
    print(f'  Coverage: {s["document_coverage"]:.1%} | '
          f'Hallucination: {s["hallucination_rate"]:.1%} | '
          f'Quality: {s["rag_system_quality"]:.3f} | '
          f'Time: {s["avg_response_time_s"]:.1f}s')

print('\n✅ k-experiment done')

In [ ]:
# ── Build summary table ───────────────────────────────────────────────────────
summaries_k = [summarize(results_k[k], f'k={k}') for k in K_VALUES]
df_k_summary = pd.DataFrame(summaries_k)

cols_show = ['label','document_coverage','hallucination_rate','faithfulness',
             'answer_relevancy','context_utilization','rag_system_quality','avg_response_time_s']
print(df_k_summary[cols_show].to_string(index=False))

In [ ]:
# ── Plot: k vs metrics ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Parameter Validation: Effect of Top-K on RAG Metrics', fontsize=13, fontweight='bold')

metrics_to_plot = [
    ('rag_system_quality',   '🧠 RAG System Quality', COLORS['bge']),
    ('document_coverage',    '📚 Document Coverage',  COLORS['ok']),
    ('hallucination_rate',   '🔴 Hallucination Rate', COLORS['warn']),
    ('avg_response_time_s',  '⏱️ Avg Response Time (s)', COLORS['accent']),
]

for ax, (metric, title, color) in zip(axes, metrics_to_plot):
    vals = df_k_summary[metric].tolist()
    bars = ax.bar([f'k={k}' for k in K_VALUES], vals, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.25 if max(vals) > 0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/param_k_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved: figures/param_k_comparison.png')

### 3.1 Interpretation

Fill in after running — key things to discuss in the report:
- At which `k` does coverage plateau?
- Does faithfulness drop as `k` increases (noise from extra chunks)?
- Is the response time increase from k=5→k=10 worth it?
- **Chosen value:** `k=5` (or justify your choice based on the plot above)

---
## 4. Experiment B — Parameter Validation: Temperature

**Hypothesis:** Lower temperature → more deterministic, more faithful answers. Higher temperature → more creative but potentially less grounded. We test `temperature ∈ {0.0, 0.3, 0.7, 1.0}` with fixed `k=5`.

In [ ]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
results_temp = {}

for temp in TEMPERATURES:
    print(f'\n── Running temperature={temp} ────────────────────────────')
    df_res = run_experiment(
        df_k_sample, vectordb_bge, emb_bge,
        k=5, temperature=temp,
        run_ragas=True
    )
    results_temp[temp] = df_res
    s = summarize(df_res, label=f'temp={temp}')
    print(f'  Faithfulness: {s["faithfulness"]:.3f} | '
          f'Relevancy: {s["answer_relevancy"]:.3f} | '
          f'Hallucination: {s["hallucination_rate"]:.1%}')

print('\n✅ Temperature experiment done')

In [ ]:
summaries_temp = [summarize(results_temp[t], f'temp={t}') for t in TEMPERATURES]
df_temp_summary = pd.DataFrame(summaries_temp)
print(df_temp_summary[cols_show].to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Parameter Validation: Effect of Temperature on RAG Quality',
             fontsize=13, fontweight='bold')

temp_metrics = [
    ('faithfulness',      '🧠 Faithfulness',      COLORS['bge']),
    ('answer_relevancy',  '💬 Answer Relevancy',  COLORS['accent']),
    ('hallucination_rate','🔴 Hallucination Rate', COLORS['warn']),
]

for ax, (metric, title, color) in zip(axes, temp_metrics):
    vals = df_temp_summary[metric].tolist()
    x_labels = [f'T={t}' for t in TEMPERATURES]
    ax.plot(x_labels, vals, marker='o', color=color, linewidth=2.5, markersize=8)
    ax.fill_between(x_labels, vals, alpha=0.15, color=color)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.05)
    for xi, yi in zip(x_labels, vals):
        ax.annotate(f'{yi:.3f}', (xi, yi), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/param_temperature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.1 Interpretation

- Does faithfulness decrease monotonically with temperature?
- Is there a sweet spot where relevancy peaks before hallucination rises?
- **Chosen value:** `temperature=0` (deterministic, maximally grounded in retrieved context)

---
## 5. Experiment C — Embedding Model Comparison

**Research question:** Does `BAAI/bge-m3` (multilingual, 570M params) outperform `all-MiniLM-L6-v2` (English-focused, 22M params) on our multilingual pediatric corpus?

⚠️ **Pre-requisite:** You must have built **two separate ChromaDB folders** — one with each embedding model. Update the paths below accordingly.

```
project/
├── chroma_db/           ← built with BAAI/bge-m3
└── chroma_db_minilm/    ← built with all-MiniLM-L6-v2
```

In [ ]:
CHROMA_BGE    = cfg.chroma_dir                      # already loaded above
CHROMA_MINILM = PROJECT_ROOT / 'chroma_db_minilm'  # adjust path if needed

# ── Load MiniLM embedding model and its vector DB ────────────────────────────
print('Loading all-MiniLM-L6-v2 vector database...')
emb_minilm = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': cfg.emb_device}
)

if CHROMA_MINILM.exists():
    vectordb_minilm = Chroma(
        persist_directory=str(CHROMA_MINILM),
        embedding_function=emb_minilm
    )
    print(f'  ✅ MiniLM DB loaded — {vectordb_minilm._collection.count()} chunks')
else:
    print(f'  ⚠️  MiniLM ChromaDB not found at {CHROMA_MINILM}')
    print('     Build it first: delete chroma_db/, change embedding_model in config.py to')
    print('     all-MiniLM-L6-v2, run the app once, then rename chroma_db/ → chroma_db_minilm/')
    vectordb_minilm = None

In [ ]:
# ── Full evaluation on all 105 questions with both models ─────────────────────
# Using best parameters found: k=5, temperature=0
BEST_K    = 5
BEST_TEMP = 0.0

print('── Running BGE-M3 (full dataset) ────────────────────────────────')
df_bge_full = run_experiment(
    df_eval, vectordb_bge, emb_bge,
    k=BEST_K, temperature=BEST_TEMP,
    run_ragas=True
)
df_bge_full.to_csv('results/results_bge_m3.csv', index=False)
print(f'  Saved → results/results_bge_m3.csv')

if vectordb_minilm is not None:
    print('\n── Running all-MiniLM-L6-v2 (full dataset) ──────────────────────')
    df_minilm_full = run_experiment(
        df_eval, vectordb_minilm, emb_minilm,
        k=BEST_K, temperature=BEST_TEMP,
        run_ragas=True
    )
    df_minilm_full.to_csv('results/results_minilm.csv', index=False)
    print(f'  Saved → results/results_minilm.csv')
else:
    print('  ⚠️  Skipping MiniLM — DB not available')
    df_minilm_full = None

In [ ]:
# ── Summary tables ────────────────────────────────────────────────────────────
s_bge    = summarize(df_bge_full,    label='BAAI/bge-m3')

if df_minilm_full is not None:
    s_minilm = summarize(df_minilm_full, label='all-MiniLM-L6-v2')
    df_emb_compare = pd.DataFrame([s_bge, s_minilm])
else:
    df_emb_compare = pd.DataFrame([s_bge])

print('\n📊 Embedding Model Comparison — Overall')
print(df_emb_compare[cols_show].to_string(index=False))

In [ ]:
# ── Per-language breakdown (EN vs ES) ─────────────────────────────────────────
def per_language_summary(df: pd.DataFrame, model_label: str) -> pd.DataFrame:
    rows = []
    for lang in df['language'].unique():
        subset = df[df['language'] == lang]
        s = summarize(subset, label=f'{model_label} [{lang}]')
        rows.append(s)
    return pd.DataFrame(rows)

df_lang_bge = per_language_summary(df_bge_full, 'BGE-M3')
print('\n📊 BGE-M3 — Per Language Breakdown')
print(df_lang_bge[['label','document_coverage','hallucination_rate',
                    'faithfulness','rag_system_quality','avg_response_time_s']].to_string(index=False))

if df_minilm_full is not None:
    df_lang_minilm = per_language_summary(df_minilm_full, 'MiniLM')
    print('\n📊 all-MiniLM — Per Language Breakdown')
    print(df_lang_minilm[['label','document_coverage','hallucination_rate',
                           'faithfulness','rag_system_quality','avg_response_time_s']].to_string(index=False))

In [ ]:
# ── Head-to-head comparison plots ─────────────────────────────────────────────
if df_minilm_full is not None:
    compare_metrics = [
        'document_coverage', 'hallucination_rate', 'faithfulness',
        'answer_relevancy', 'context_utilization', 'rag_system_quality'
    ]
    labels_plot = ['Coverage', 'Hallucination\nRate', 'Faithfulness',
                   'Answer\nRelevancy', 'Context\nUtilization', 'RAG\nQuality']

    bge_vals    = [s_bge[m]    for m in compare_metrics]
    minilm_vals = [s_minilm[m] for m in compare_metrics]

    x = np.arange(len(compare_metrics))
    w = 0.35

    fig, ax = plt.subplots(figsize=(13, 5))
    b1 = ax.bar(x - w/2, bge_vals,    w, label='BAAI/bge-m3',       color=COLORS['bge'],    alpha=0.88, edgecolor='white')
    b2 = ax.bar(x + w/2, minilm_vals, w, label='all-MiniLM-L6-v2',  color=COLORS['minilm'], alpha=0.88, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(labels_plot, fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_title('Embedding Model Comparison: BGE-M3 vs all-MiniLM-L6-v2',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylabel('Score')

    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.02, f'{h:.2f}',
                ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig('figures/embedding_model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

else:
    print('ℹ️  Skipping head-to-head plot — MiniLM results not available')

In [ ]:
# ── Language-specific comparison: EN vs ES for each model ────────────────────
if df_minilm_full is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Multilingual Performance: English vs Spanish queries',
                 fontsize=13, fontweight='bold')

    for ax, (df_res, model_label, color) in zip(axes, [
        (df_bge_full,    'BAAI/bge-m3',      COLORS['bge']),
        (df_minilm_full, 'all-MiniLM-L6-v2', COLORS['minilm']),
    ]):
        en_cov = summarize(df_res[df_res['language']=='English'],  'EN')['document_coverage']
        es_cov = summarize(df_res[df_res['language']=='Spanish'],  'ES')['document_coverage']
        en_q   = summarize(df_res[df_res['language']=='English'],  'EN')['rag_system_quality']
        es_q   = summarize(df_res[df_res['language']=='Spanish'],  'ES')['rag_system_quality']

        bars = ax.bar(['EN Coverage','ES Coverage','EN Quality','ES Quality'],
                      [en_cov, es_cov, en_q, es_q],
                      color=[color, color+'99', color, color+'99'],
                      edgecolor='white')
        ax.set_title(model_label, fontsize=11, fontweight='bold')
        ax.set_ylim(0, 1.2)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.02, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('figures/multilingual_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Final Results: Confusion Matrix & Classification Report

In [ ]:
def plot_confusion_matrix(df: pd.DataFrame, title: str, ax, color):
    TP = int(((df['label']==1)&(df['pred']==1)).sum())
    FP = int(((df['label']==0)&(df['pred']==1)).sum())
    TN = int(((df['label']==0)&(df['pred']==0)).sum())
    FN = int(((df['label']==1)&(df['pred']==0)).sum())
    cm = np.array([[TP, FN],[FP, TN]])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap=sns.light_palette(color, as_cmap=True),
                xticklabels=['Pred: Answered','Pred: Refused'],
                yticklabels=['Actual: In DB','Actual: Not in DB'])
    ax.set_title(title, fontsize=11, fontweight='bold')

ncols = 2 if df_minilm_full is not None else 1
fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 4))
if ncols == 1:
    axes = [axes]

plot_confusion_matrix(df_bge_full, 'BGE-M3 (final)', axes[0], COLORS['bge'])
if df_minilm_full is not None:
    plot_confusion_matrix(df_minilm_full, 'all-MiniLM-L6-v2 (final)', axes[1], COLORS['minilm'])

plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final classification summary ──────────────────────────────────────────────
final_rows = [s_bge]
if df_minilm_full is not None:
    final_rows.append(s_minilm)

df_final = pd.DataFrame(final_rows)
class_cols = ['label','accuracy','precision','recall','f1',
              'document_coverage','hallucination_rate','rag_system_quality','avg_response_time_s']

print('\n📊 FINAL RESULTS SUMMARY')
print('='*80)
print(df_final[class_cols].to_string(index=False))

df_final.to_csv('results/final_comparison_summary.csv', index=False)
print('\n✅ Saved → results/final_comparison_summary.csv')

---
## 7. Qualitative Analysis — Failure Cases

In [ ]:
# ── False Negatives: in-DB questions the model refused to answer ──────────────
print('=== FALSE NEGATIVES (Miss Rate) ===')
fn_bge = df_bge_full[(df_bge_full['label']==1) & (df_bge_full['pred']==0)]
print(f'BGE-M3: {len(fn_bge)} questions missed')
if len(fn_bge):
    display(fn_bge[['id','question','language','answer_preview']].head(5))

print('\n=== FALSE POSITIVES (Hallucination Flags) ===')
fp_bge = df_bge_full[(df_bge_full['label']==0) & (df_bge_full['pred']==1)]
print(f'BGE-M3: {len(fp_bge)} out-of-scope questions answered')
if len(fp_bge):
    display(fp_bge[['id','question','language','answer_preview']].head(5))

In [ ]:
# ── Distribution of response times ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_bge_full['response_time'], bins=20, color=COLORS['bge'],
        alpha=0.8, edgecolor='white', label='BGE-M3')
if df_minilm_full is not None:
    ax.hist(df_minilm_full['response_time'], bins=20, color=COLORS['minilm'],
            alpha=0.6, edgecolor='white', label='all-MiniLM')
ax.axvline(df_bge_full['response_time'].mean(), color=COLORS['bge'],
           linestyle='--', linewidth=2, label=f'BGE mean={df_bge_full["response_time"].mean():.1f}s')
ax.set_xlabel('Response Time (s)')
ax.set_ylabel('Count')
ax.set_title('Response Time Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('figures/response_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Summary for the Report

Run this cell last to print a clean copy-paste summary for the paper.

In [ ]:
print('='*70)
print('BABYCHATTY RAG SYSTEM — FINAL EVALUATION SUMMARY'.center(70))
print('='*70)

print('\n[1] CHOSEN PARAMETERS')
print(f'  - Top-K retrieval    : k=5')
print(f'  - Temperature        : 0.0 (deterministic)')
print(f'  - Chunk size         : {cfg.chunk_size} chars / {cfg.chunk_overlap} overlap')
print(f'  - Embedding model    : {cfg.embedding_model}')
print(f'  - LLM                : {cfg.model_name}')

print('\n[2] SYSTEM PERFORMANCE (BGE-M3, k=5, T=0)')
print(f'  - Document Coverage  : {s_bge["document_coverage"]:.1%}')
print(f'  - Hallucination Rate : {s_bge["hallucination_rate"]:.1%}')
print(f'  - Avg Response Time  : {s_bge["avg_response_time_s"]:.1f}s')
print(f'  - RAG System Quality : {s_bge["rag_system_quality"]:.3f}')
print(f'     Faithfulness       : {s_bge["faithfulness"]:.3f}')
print(f'     Answer Relevancy   : {s_bge["answer_relevancy"]:.3f}')
print(f'     Context Utilization: {s_bge["context_utilization"]:.3f}')
print(f'  - Accuracy           : {s_bge["accuracy"]:.3f}')
print(f'  - F1 Score           : {s_bge["f1"]:.3f}')

if df_minilm_full is not None:
    print('\n[3] EMBEDDING MODEL COMPARISON')
    for s, name in [(s_bge, 'BAAI/bge-m3'), (s_minilm, 'all-MiniLM-L6-v2')]:
        print(f'  {name}')
        print(f'    Coverage={s["document_coverage"]:.1%} | '
              f'Hallucination={s["hallucination_rate"]:.1%} | '
              f'Quality={s["rag_system_quality"]:.3f} | '
              f'Time={s["avg_response_time_s"]:.1f}s')

print('\n' + '='*70)

---
## Appendix: Output Files

| File | Description |
|---|---|
| `results/results_bge_m3.csv` | Per-question results with BGE-M3 |
| `results/results_minilm.csv` | Per-question results with all-MiniLM |
| `results/final_comparison_summary.csv` | Aggregate metrics both models |
| `figures/param_k_comparison.png` | k ablation plots |
| `figures/param_temperature_comparison.png` | Temperature ablation plots |
| `figures/embedding_model_comparison.png` | Head-to-head model comparison |
| `figures/multilingual_comparison.png` | EN vs ES breakdown |
| `figures/confusion_matrices.png` | Confusion matrices |
| `figures/response_time_distribution.png` | Response time histograms |